# Live Database Fetcher

This notebook fetches SQLite databases and YAML configuration files from the remote Hummingbot server.

**Server Path Structure:**
```
root/deploy/hummingbot-api/bots/instances/
    ├── bot_instance_1/
    │   ├── conf/
    │   │   └── controllers/  (*.yml files)
    │   ├── data/            (*.sqlite files)
    │   └── logs/
    ├── bot_instance_2/
    └── ...
```

**What it fetches:**
- All bot instance directories from `instances/`
- SQLite databases from `data/` directory
- YAML config files from `conf/controllers/` directory

**Local Storage:**
- Downloads to: `data/live_databases/<bot_name>/`
- Replaces old databases when fetching updates
- Preserves directory structure: `data/` and `conf/controllers/`

In [10]:
import subprocess
import os
from datetime import datetime
from pathlib import Path
import logging
from typing import List, Dict, Optional
import json

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [11]:
# ============================================================================
#                            CONFIGURATION
# ============================================================================

# SSH Configuration
SSH_HOST = "brigado"  # Your SSH host alias
REMOTE_BASE_PATH = "deploy/hummingbot-api/bots/instances"

# Local Configuration
LOCAL_BASE_PATH = Path("/Users/tomasgaudino/PycharmProjects/quants-lab/research_notebooks/brigado_v2/data/live_databases")
LOCAL_BASE_PATH.mkdir(parents=True, exist_ok=True)

# Log file - saved at root level with timestamp
FETCH_TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H%M%S")
LOG_FILE = LOCAL_BASE_PATH / f"fetch_log_{FETCH_TIMESTAMP}.json"

print(f"Fetch Timestamp: {FETCH_TIMESTAMP}")
print(f"Local Destination: {LOCAL_BASE_PATH}")
print(f"Log File: {LOG_FILE}")

Fetch Timestamp: 2026-03-04_213300
Local Destination: /Users/tomasgaudino/PycharmProjects/quants-lab/research_notebooks/brigado_v2/data/live_databases
Log File: /Users/tomasgaudino/PycharmProjects/quants-lab/research_notebooks/brigado_v2/data/live_databases/fetch_log_2026-03-04_213300.json


In [12]:
# ============================================================================
#                         SSH UTILITY FUNCTIONS
# ============================================================================

def run_ssh_command(command: str) -> tuple[int, str, str]:
    """
    Execute command on remote server via SSH.
    
    Returns:
        tuple: (return_code, stdout, stderr)
    """
    full_command = f"ssh {SSH_HOST} '{command}'"
    logger.info(f"Executing: {full_command}")
    
    try:
        result = subprocess.run(
            full_command,
            shell=True,
            capture_output=True,
            text=True,
            timeout=30
        )
        return result.returncode, result.stdout, result.stderr
    except subprocess.TimeoutExpired:
        logger.error(f"Command timed out: {command}")
        return -1, "", "Command timed out"
    except Exception as e:
        logger.error(f"Command failed: {e}")
        return -1, "", str(e)


def list_remote_directories(remote_path: str) -> List[str]:
    """
    List all directories in remote path.
    
    Returns:
        List of directory names
    """
    command = f"ls -1 {remote_path}"
    returncode, stdout, stderr = run_ssh_command(command)
    
    if returncode != 0:
        logger.error(f"Failed to list directories: {stderr}")
        return []
    
    directories = [d.strip() for d in stdout.split('\n') if d.strip()]
    logger.info(f"Found {len(directories)} directories in {remote_path}")
    return directories


def scp_download(remote_path: str, local_path: Path, recursive: bool = False) -> bool:
    """
    Download file or directory from remote server using scp.
    
    Returns:
        bool: True if successful, False otherwise
    """
    local_path.parent.mkdir(parents=True, exist_ok=True)
    
    recursive_flag = "-r" if recursive else ""
    command = f"scp {recursive_flag} {SSH_HOST}:{remote_path} {local_path}"
    logger.info(f"Downloading: {remote_path} -> {local_path}")
    
    try:
        result = subprocess.run(
            command,
            shell=True,
            capture_output=True,
            text=True,
            timeout=300  # 5 minutes for large files
        )
        
        if result.returncode == 0:
            logger.info(f"✓ Downloaded: {remote_path}")
            return True
        else:
            logger.error(f"✗ Failed to download {remote_path}: {result.stderr}")
            return False
    except Exception as e:
        logger.error(f"✗ Exception during download: {e}")
        return False


def check_remote_path_exists(remote_path: str) -> bool:
    """
    Check if remote path exists.
    """
    command = f"test -e {remote_path} && echo 'exists' || echo 'not_found'"
    returncode, stdout, stderr = run_ssh_command(command)
    return 'exists' in stdout.lower()


print("✓ SSH utility functions loaded")

✓ SSH utility functions loaded


In [13]:
# ============================================================================
#                      DATABASE FETCH FUNCTIONS
# ============================================================================

def fetch_bot_instance_data(bot_name: str) -> Dict[str, any]:
    """
    Fetch SQLite database and YAML configs for a specific bot instance.
    Replaces old files if they exist.
    
    Returns:
        Dict with fetch results
    """
    logger.info(f"\n{'='*60}")
    logger.info(f"Processing bot instance: {bot_name}")
    logger.info(f"{'='*60}")
    
    result = {
        'bot_name': bot_name,
        'timestamp': datetime.now().isoformat(),
        'databases': [],
        'configs': [],
        'errors': [],
        'replaced_files': []
    }
    
    # Create local directory for this bot (directly under live_databases/)
    bot_local_path = LOCAL_BASE_PATH / bot_name
    bot_local_path.mkdir(parents=True, exist_ok=True)
    
    # ========================================================================
    # 1. Fetch SQLite databases from data/ directory
    # ========================================================================
    remote_data_path = f"{REMOTE_BASE_PATH}/{bot_name}/data"
    
    if check_remote_path_exists(remote_data_path):
        logger.info(f"\n[1/2] Fetching SQLite databases from {remote_data_path}")
        
        # List .sqlite files
        command = f"ls -1 {remote_data_path}/*.sqlite 2>/dev/null || echo 'no_sqlite_files'"
        returncode, stdout, stderr = run_ssh_command(command)
        
        if 'no_sqlite_files' not in stdout:
            sqlite_files = [f.strip() for f in stdout.split('\n') if f.strip() and f.endswith('.sqlite')]
            
            for sqlite_file in sqlite_files:
                filename = os.path.basename(sqlite_file)
                local_db_path = bot_local_path / "data" / filename
                local_db_path.parent.mkdir(parents=True, exist_ok=True)
                
                # Check if file exists (will be replaced)
                file_existed = local_db_path.exists()
                if file_existed:
                    old_size = local_db_path.stat().st_size
                    result['replaced_files'].append({
                        'filename': filename,
                        'type': 'database',
                        'old_size_mb': round(old_size / (1024 * 1024), 2)
                    })
                    logger.info(f"  Replacing existing database: {filename}")
                
                success = scp_download(sqlite_file, local_db_path)
                
                if success:
                    file_size = local_db_path.stat().st_size
                    result['databases'].append({
                        'filename': filename,
                        'local_path': str(local_db_path),
                        'size_bytes': file_size,
                        'size_mb': round(file_size / (1024 * 1024), 2),
                        'replaced': file_existed
                    })
                else:
                    result['errors'].append(f"Failed to download {filename}")
        else:
            logger.warning(f"  No .sqlite files found in {remote_data_path}")
            result['errors'].append("No .sqlite files found")
    else:
        logger.warning(f"  Data directory does not exist: {remote_data_path}")
        result['errors'].append("Data directory not found")
    
    # ========================================================================
    # 2. Fetch YAML configs from conf/controllers/ directory
    # ========================================================================
    remote_config_path = f"{REMOTE_BASE_PATH}/{bot_name}/conf/controllers"
    
    if check_remote_path_exists(remote_config_path):
        logger.info(f"\n[2/2] Fetching YAML configs from {remote_config_path}")
        
        # List .yml files
        command = f"ls -1 {remote_config_path}/*.yml 2>/dev/null || echo 'no_yml_files'"
        returncode, stdout, stderr = run_ssh_command(command)
        
        if 'no_yml_files' not in stdout:
            yml_files = [f.strip() for f in stdout.split('\n') if f.strip() and f.endswith('.yml')]
            
            for yml_file in yml_files:
                filename = os.path.basename(yml_file)
                local_config_path = bot_local_path / "conf" / "controllers" / filename
                local_config_path.parent.mkdir(parents=True, exist_ok=True)
                
                # Check if file exists (will be replaced)
                file_existed = local_config_path.exists()
                if file_existed:
                    result['replaced_files'].append({
                        'filename': filename,
                        'type': 'config'
                    })
                    logger.info(f"  Replacing existing config: {filename}")
                
                success = scp_download(yml_file, local_config_path)
                
                if success:
                    file_size = local_config_path.stat().st_size
                    result['configs'].append({
                        'filename': filename,
                        'local_path': str(local_config_path),
                        'size_bytes': file_size,
                        'replaced': file_existed
                    })
                else:
                    result['errors'].append(f"Failed to download {filename}")
        else:
            logger.warning(f"  No .yml files found in {remote_config_path}")
            result['errors'].append("No .yml files found")
    else:
        logger.warning(f"  Config directory does not exist: {remote_config_path}")
        result['errors'].append("Config directory not found")
    
    logger.info(f"\n✓ Bot instance '{bot_name}' processed:")
    logger.info(f"  - Databases: {len(result['databases'])} ({sum(1 for d in result['databases'] if d.get('replaced', False))} replaced)")
    logger.info(f"  - Configs: {len(result['configs'])} ({sum(1 for c in result['configs'] if c.get('replaced', False))} replaced)")
    logger.info(f"  - Errors: {len(result['errors'])}")
    
    return result


def fetch_all_instances() -> Dict[str, any]:
    """
    Fetch data from all bot instances on the server.
    
    Returns:
        Dict with complete fetch summary
    """
    logger.info(f"\n{'='*80}")
    logger.info("STARTING LIVE DATABASE FETCH")
    logger.info(f"{'='*80}")
    logger.info(f"Remote Server: {SSH_HOST}")
    logger.info(f"Remote Path: {REMOTE_BASE_PATH}")
    logger.info(f"Local Path: {LOCAL_BASE_PATH}")
    logger.info(f"{'='*80}\n")
    
    # Get list of all bot instances
    bot_instances = list_remote_directories(REMOTE_BASE_PATH)
    
    if not bot_instances:
        logger.error("No bot instances found on server!")
        return {
            'status': 'failed',
            'error': 'No bot instances found',
            'timestamp': datetime.now().isoformat()
        }
    
    logger.info(f"Found {len(bot_instances)} bot instance(s):")
    for bot in bot_instances:
        logger.info(f"  - {bot}")
    
    # Fetch data for each bot instance
    results = []
    for bot_name in bot_instances:
        try:
            result = fetch_bot_instance_data(bot_name)
            results.append(result)
        except Exception as e:
            logger.error(f"Exception processing {bot_name}: {e}")
            results.append({
                'bot_name': bot_name,
                'timestamp': datetime.now().isoformat(),
                'databases': [],
                'configs': [],
                'errors': [str(e)],
                'replaced_files': []
            })
    
    # Calculate totals
    total_databases = sum(len(r['databases']) for r in results)
    total_configs = sum(len(r['configs']) for r in results)
    total_replaced = sum(len(r['replaced_files']) for r in results)
    
    # Create summary
    summary = {
        'status': 'completed',
        'timestamp': datetime.now().isoformat(),
        'fetch_id': FETCH_TIMESTAMP,
        'ssh_host': SSH_HOST,
        'remote_base_path': REMOTE_BASE_PATH,
        'local_base_path': str(LOCAL_BASE_PATH),
        'total_instances': len(bot_instances),
        'total_databases': total_databases,
        'total_configs': total_configs,
        'total_errors': sum(len(r['errors']) for r in results),
        'total_replaced_files': total_replaced,
        'instances': results
    }
    
    # Save log to JSON
    with open(LOG_FILE, 'w') as f:
        json.dump(summary, f, indent=2)
    
    logger.info(f"\n{'='*80}")
    logger.info("FETCH COMPLETE")
    logger.info(f"{'='*80}")
    logger.info(f"Total Instances: {summary['total_instances']}")
    logger.info(f"Total Databases: {summary['total_databases']}")
    logger.info(f"Total Configs: {summary['total_configs']}")
    logger.info(f"Total Replaced Files: {summary['total_replaced_files']}")
    logger.info(f"Total Errors: {summary['total_errors']}")
    logger.info(f"Log saved to: {LOG_FILE}")
    logger.info(f"{'='*80}\n")
    
    return summary


print("✓ Database fetch functions loaded")

✓ Database fetch functions loaded


In [14]:
# ============================================================================
#                          EXECUTE FETCH
# ============================================================================

# Run the fetch operation
fetch_summary = fetch_all_instances()

2026-03-04 21:33:00,907 - INFO - 
2026-03-04 21:33:00,907 - INFO - STARTING LIVE DATABASE FETCH
2026-03-04 21:33:00,907 - INFO - ================================================================================
2026-03-04 21:33:00,907 - INFO - Remote Server: brigado
2026-03-04 21:33:00,907 - INFO - Remote Path: deploy/hummingbot-api/bots/instances
2026-03-04 21:33:00,908 - INFO - Local Path: /Users/tomasgaudino/PycharmProjects/quants-lab/research_notebooks/brigado_v2/data/live_databases
2026-03-04 21:33:00,908 - INFO - ================================================================================

2026-03-04 21:33:00,908 - INFO - Executing: ssh brigado 'ls -1 deploy/hummingbot-api/bots/instances'
2026-03-04 21:33:07,514 - INFO - Found 2 directories in deploy/hummingbot-api/bots/instances
2026-03-04 21:33:07,515 - INFO - Found 2 bot instance(s):
2026-03-04 21:33:07,516 - INFO -   - pmm-btcbrl-20260303-012028
2026-03-04 21:33:07,516 - INFO -   - pmm-mister-all-20260303-011107
2026-03-04

In [15]:
# ============================================================================
#                        DISPLAY DETAILED SUMMARY
# ============================================================================

import pandas as pd

print("\n" + "="*80)
print("FETCH SUMMARY")
print("="*80)

print(f"\nFetch ID: {fetch_summary['fetch_id']}")
print(f"Status: {fetch_summary['status']}")
print(f"Local Path: {fetch_summary['local_base_path']}")

print(f"\nOverall Statistics:")
print(f"  Total Bot Instances: {fetch_summary['total_instances']}")
print(f"  Total Databases: {fetch_summary['total_databases']}")
print(f"  Total Config Files: {fetch_summary['total_configs']}")
print(f"  Total Replaced Files: {fetch_summary.get('total_replaced_files', 0)}")
print(f"  Total Errors: {fetch_summary['total_errors']}")

print(f"\n{'-'*80}")
print("Per-Instance Details:")
print("-"*80)

for instance in fetch_summary['instances']:
    print(f"\n🤖 Bot: {instance['bot_name']}")
    
    if instance['databases']:
        print(f"\n  📊 Databases ({len(instance['databases'])}):")
        for db in instance['databases']:
            replaced_marker = " [REPLACED]" if db.get('replaced', False) else " [NEW]"
            print(f"    ✓ {db['filename']} ({db['size_mb']} MB){replaced_marker}")
            print(f"      Path: {db['local_path']}")
    else:
        print("  📊 Databases: None found")
    
    if instance['configs']:
        print(f"\n  ⚙️  Configs ({len(instance['configs'])}):")
        for cfg in instance['configs']:
            replaced_marker = " [REPLACED]" if cfg.get('replaced', False) else " [NEW]"
            print(f"    ✓ {cfg['filename']}{replaced_marker}")
            print(f"      Path: {cfg['local_path']}")
    else:
        print("  ⚙️  Configs: None found")
    
    if instance['errors']:
        print(f"\n  ⚠️  Errors ({len(instance['errors'])}):")
        for error in instance['errors']:
            print(f"    ✗ {error}")

print(f"\n{'='*80}")

# Create DataFrame for easier analysis
db_records = []
for instance in fetch_summary['instances']:
    for db in instance['databases']:
        db_records.append({
            'bot_name': instance['bot_name'],
            'filename': db['filename'],
            'size_mb': db['size_mb'],
            'replaced': db.get('replaced', False),
            'local_path': db['local_path']
        })

if db_records:
    df_databases = pd.DataFrame(db_records)
    print("\nDatabases DataFrame:")
    print(df_databases[['bot_name', 'filename', 'size_mb', 'replaced']])
    
    print(f"\nTotal Database Size: {df_databases['size_mb'].sum():.2f} MB")
    print(f"Databases Replaced: {df_databases['replaced'].sum()}")
    print(f"New Databases: {(~df_databases['replaced']).sum()}")
else:
    print("\nNo databases fetched.")


FETCH SUMMARY

Fetch ID: 2026-03-04_213300
Status: completed
Local Path: /Users/tomasgaudino/PycharmProjects/quants-lab/research_notebooks/brigado_v2/data/live_databases

Overall Statistics:
  Total Bot Instances: 2
  Total Databases: 2
  Total Config Files: 10
  Total Replaced Files: 0
  Total Errors: 0

--------------------------------------------------------------------------------
Per-Instance Details:
--------------------------------------------------------------------------------

🤖 Bot: pmm-btcbrl-20260303-012028

  📊 Databases (1):
    ✓ pmm-btcbrl-20260303-012028.sqlite (42.45 MB) [NEW]
      Path: /Users/tomasgaudino/PycharmProjects/quants-lab/research_notebooks/brigado_v2/data/live_databases/pmm-btcbrl-20260303-012028/data/pmm-btcbrl-20260303-012028.sqlite

  ⚙️  Configs (5):
    ✓ brigado-binance-btcbrl-1.yml [NEW]
      Path: /Users/tomasgaudino/PycharmProjects/quants-lab/research_notebooks/brigado_v2/data/live_databases/pmm-btcbrl-20260303-012028/conf/controllers/brigado

In [16]:
# ============================================================================
#                        VERIFY DOWNLOADED FILES
# ============================================================================

print("\nVerifying downloaded files...\n")

for instance in fetch_summary['instances']:
    bot_name = instance['bot_name']
    print(f"Bot: {bot_name}")
    
    # Check databases
    for db in instance['databases']:
        db_path = Path(db['local_path'])
        if db_path.exists():
            print(f"  ✓ Database exists: {db['filename']}")
            
            # Try to verify it's a valid SQLite file
            try:
                import sqlite3
                conn = sqlite3.connect(db_path)
                cursor = conn.cursor()
                cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
                tables = cursor.fetchall()
                conn.close()
                print(f"    Tables: {len(tables)} found")
            except Exception as e:
                print(f"    ⚠️  Warning: Could not verify SQLite file: {e}")
        else:
            print(f"  ✗ Database missing: {db['filename']}")
    
    # Check configs
    for cfg in instance['configs']:
        cfg_path = Path(cfg['local_path'])
        if cfg_path.exists():
            print(f"  ✓ Config exists: {cfg['filename']}")
        else:
            print(f"  ✗ Config missing: {cfg['filename']}")
    
    print()

print("Verification complete!")


Verifying downloaded files...

Bot: pmm-btcbrl-20260303-012028
  ✓ Database exists: pmm-btcbrl-20260303-012028.sqlite
    Tables: 13 found
  ✓ Config exists: brigado-binance-btcbrl-1.yml
  ✓ Config exists: brigado-binance-btcbrl-2.yml
  ✓ Config exists: brigado-binance-btcbrl-3.yml
  ✓ Config exists: brigado-binance-btcbrl-4.yml
  ✓ Config exists: brigado-binance-btcbrl-5.yml

Bot: pmm-mister-all-20260303-011107
  ✓ Database exists: pmm-mister-all-20260303-011107.sqlite
    Tables: 13 found
  ✓ Config exists: brigado-binance-18-1.yml
  ✓ Config exists: brigado-binance-21-1.yml
  ✓ Config exists: brigado-binance-21-2.yml
  ✓ Config exists: brigado-binance-21-3.yml
  ✓ Config exists: brigado-binance-21-4.yml

Verification complete!


In [17]:
# ============================================================================#                   DATABASE CORRUPTION CHECK & RECOVERY# ============================================================================print("\n" + "="*80)print("DATABASE CORRUPTION CHECK & RECOVERY")print("="*80 + "\n")import sqlite3import subprocesscorruption_log = {    'timestamp': datetime.now().isoformat(),    'databases_checked': 0,    'databases_corrupted': 0,    'databases_recovered': 0,    'recovery_failures': 0,    'details': []}for instance in fetch_summary['instances']:    bot_name = instance['bot_name']        for db in instance['databases']:        db_path = Path(db['local_path'])                if not db_path.exists():            continue                corruption_log['databases_checked'] += 1                print(f"Checking: {db['filename']}")                # Run SQLite integrity check        try:            check_cmd = f'sqlite3 "{db_path}" "PRAGMA integrity_check;"'            result = subprocess.run(check_cmd, shell=True, capture_output=True, text=True, timeout=30)                        is_corrupt = False            if result.returncode != 0 or 'ok' not in result.stdout.lower():                is_corrupt = True                print(f"  ⚠️  CORRUPTION DETECTED!")                corruption_log['databases_corrupted'] += 1            else:                print(f"  ✓ Database integrity OK")                corruption_log['details'].append({                    'database': db['filename'],                    'bot_name': bot_name,                    'status': 'ok',                    'corrupted': False,                    'recovered': False                })                continue                        if is_corrupt:                # Attempt recovery                print(f"  🔧 Attempting recovery...")                                backup_path = db_path.parent / f"{db_path.stem}_corrupted_backup.sqlite"                recovered_path = db_path.parent / f"{db_path.stem}_recovered.sqlite"                                # Run .recover command                dump_cmd = f'sqlite3 "{db_path}" ".recover"'                restore_cmd = f'sqlite3 "{recovered_path}"'                                try:                    dump_proc = subprocess.Popen(dump_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)                    restore_proc = subprocess.Popen(restore_cmd, shell=True, stdin=dump_proc.stdout, stdout=subprocess.PIPE, stderr=subprocess.PIPE)                    dump_proc.stdout.close()                                        stdout, stderr = restore_proc.communicate(timeout=120)                                        if restore_proc.returncode == 0 and recovered_path.exists():                        # Verify recovered database                        verify_cmd = f'sqlite3 "{recovered_path}" "SELECT COUNT(*) FROM Executors WHERE net_pnl_quote != 0;"'                        verify_result = subprocess.run(verify_cmd, shell=True, capture_output=True, text=True, timeout=30)                                                if verify_result.returncode == 0:                            executor_count = verify_result.stdout.strip()                            print(f"  ✓ Recovery successful!")                            print(f"    - Verified: {executor_count} executors with non-zero PnL")                                                        # Replace corrupted with recovered, then delete backup                            db_path.rename(backup_path)                            recovered_path.rename(db_path)                            backup_path.unlink()  # Delete corrupted backup                            print(f"    - Removed corrupted backup: {backup_path.name}")                                                        corruption_log['databases_recovered'] += 1                            corruption_log['details'].append({                                'database': db['filename'],                                'bot_name': bot_name,                                'status': 'recovered',                                'corrupted': True,                                'recovered': True,                                'backup_path': str(backup_path),                                'executors_with_pnl': executor_count                            })                        else:                            print(f"  ✗ Recovery verification failed: {verify_result.stderr}")                            corruption_log['recovery_failures'] += 1                            corruption_log['details'].append({                                'database': db['filename'],                                'bot_name': bot_name,                                'status': 'recovery_failed',                                'corrupted': True,                                'recovered': False,                                'error': verify_result.stderr                            })                    else:                        print(f"  ✗ Recovery failed: {stderr.decode()}")                        corruption_log['recovery_failures'] += 1                        corruption_log['details'].append({                            'database': db['filename'],                            'bot_name': bot_name,                            'status': 'recovery_failed',                            'corrupted': True,                            'recovered': False,                            'error': stderr.decode() if stderr else 'Unknown error'                        })                                        except subprocess.TimeoutExpired:                    print(f"  ✗ Recovery timed out")                    corruption_log['recovery_failures'] += 1                    corruption_log['details'].append({                        'database': db['filename'],                        'bot_name': bot_name,                        'status': 'recovery_timeout',                        'corrupted': True,                        'recovered': False,                        'error': 'Recovery timeout'                    })                except Exception as e:                    print(f"  ✗ Recovery exception: {e}")                    corruption_log['recovery_failures'] += 1                    corruption_log['details'].append({                        'database': db['filename'],                        'bot_name': bot_name,                        'status': 'recovery_exception',                        'corrupted': True,                        'recovered': False,                        'error': str(e)                    })                except Exception as e:            print(f"  ✗ Integrity check failed: {e}")            corruption_log['details'].append({                'database': db['filename'],                'bot_name': bot_name,                'status': 'check_failed',                'corrupted': 'unknown',                'recovered': False,                'error': str(e)            })print(f"\n{'='*80}")print("CORRUPTION CHECK SUMMARY")print(f"{'='*80}")print(f"Databases checked: {corruption_log['databases_checked']}")print(f"Corrupted: {corruption_log['databases_corrupted']}")print(f"Successfully recovered: {corruption_log['databases_recovered']}")print(f"Recovery failures: {corruption_log['recovery_failures']}")print(f"{'='*80}\n")# Save corruption logcorruption_log_file = LOCAL_BASE_PATH / f"corruption_log_{FETCH_TIMESTAMP}.json"with open(corruption_log_file, 'w') as f:    json.dump(corruption_log, f, indent=2)print(f"Corruption log saved to: {corruption_log_file}")

In [18]:
# ============================================================================
#                TRADE-TO-CONTROLLER MAPPING VERIFICATION
# ============================================================================

print("\n" + "="*80)
print("TRADE-TO-CONTROLLER MAPPING VERIFICATION")
print("="*80 + "\n")

import pandas as pd
import numpy as np
import json

mapping_log = {
    'timestamp': datetime.now().isoformat(),
    'databases_tested': 0,
    'mapping_results': []
}

for instance in fetch_summary['instances']:
    bot_name = instance['bot_name']
    
    for db in instance['databases']:
        db_path = Path(db['local_path'])
        
        if not db_path.exists():
            continue
        
        mapping_log['databases_tested'] += 1
        
        print(f"Testing: {db['filename']}")
        
        try:
            conn = sqlite3.connect(db_path)
            
            # Load trades
            trades = pd.read_sql_query("SELECT order_id FROM TradeFill", conn)
            total_trades = len(trades)
            print(f"  Trades: {total_trades}")
            
            # Load executors and parse custom_info
            executors = pd.read_sql_query(
                "SELECT controller_id, custom_info, net_pnl_quote FROM Executors WHERE net_pnl_quote != 0", 
                conn
            )
            print(f"  Executors (with PnL): {len(executors)}")
            
            # Parse custom_info and create mapping
            order_to_controller = {}
            executors_with_order_ids = 0
            
            for _, executor in executors.iterrows():
                controller_id = executor['controller_id']
                custom_info_str = executor['custom_info']
                
                if custom_info_str:
                    try:
                        custom_info = json.loads(custom_info_str)
                        order_ids = custom_info.get('order_ids', [])
                        
                        if order_ids:
                            executors_with_order_ids += 1
                            if isinstance(order_ids, list):
                                for oid in order_ids:
                                    if oid:
                                        order_to_controller[str(oid)] = controller_id
                    except:
                        pass
            
            print(f"  Executors with order_ids: {executors_with_order_ids}")
            print(f"  Total order mappings: {len(order_to_controller)}")
            
            # Test mapping
            trades['controller_id'] = trades['order_id'].map(order_to_controller)
            mapped_trades = trades['controller_id'].notna().sum()
            coverage_pct = (mapped_trades / total_trades * 100) if total_trades > 0 else 0
            
            print(f"  Mapped trades: {mapped_trades}/{total_trades} ({coverage_pct:.1f}%)")
            
            # Get controller breakdown
            controller_counts = trades[trades['controller_id'].notna()]['controller_id'].value_counts().to_dict()
            
            status = 'excellent' if coverage_pct >= 90 else 'good' if coverage_pct >= 70 else 'fair' if coverage_pct >= 50 else 'poor'
            
            mapping_log['mapping_results'].append({
                'database': db['filename'],
                'bot_name': bot_name,
                'total_trades': int(total_trades),
                'total_executors': int(len(executors)),
                'executors_with_order_ids': int(executors_with_order_ids),
                'total_order_mappings': len(order_to_controller),
                'mapped_trades': int(mapped_trades),
                'unmapped_trades': int(total_trades - mapped_trades),
                'coverage_percent': round(coverage_pct, 2),
                'status': status,
                'controller_breakdown': controller_counts
            })
            
            if coverage_pct >= 70:
                print(f"  ✓ Status: {status.upper()}")
            else:
                print(f"  ⚠️  Status: {status.upper()}")
            
            print(f"\n  Controller breakdown:")
            for ctrl, count in sorted(controller_counts.items(), key=lambda x: x[1], reverse=True):
                print(f"    - {ctrl}: {count} trades")
            
            conn.close()
            
        except Exception as e:
            print(f"  ✗ Mapping test failed: {e}")
            mapping_log['mapping_results'].append({
                'database': db['filename'],
                'bot_name': bot_name,
                'status': 'error',
                'error': str(e)
            })
        
        print()

print(f"{'='*80}")
print("MAPPING VERIFICATION SUMMARY")
print(f"{'='*80}")

total_trades_all = sum(r.get('total_trades', 0) for r in mapping_log['mapping_results'])
total_mapped_all = sum(r.get('mapped_trades', 0) for r in mapping_log['mapping_results'])
overall_coverage = (total_mapped_all / total_trades_all * 100) if total_trades_all > 0 else 0

print(f"Databases tested: {mapping_log['databases_tested']}")
print(f"Total trades: {total_trades_all}")
print(f"Total mapped: {total_mapped_all}")
print(f"Overall coverage: {overall_coverage:.1f}%")
print(f"{'='*80}\n")

# Save mapping log
mapping_log_file = LOCAL_BASE_PATH / f"mapping_verification_{FETCH_TIMESTAMP}.json"
with open(mapping_log_file, 'w') as f:
    json.dump(mapping_log, f, indent=2)

print(f"Mapping verification log saved to: {mapping_log_file}")


TRADE-TO-CONTROLLER MAPPING VERIFICATION

Testing: pmm-btcbrl-20260303-012028.sqlite
  Trades: 10580
  Executors (with PnL): 4301
  Executors with order_ids: 4301
  Total order mappings: 8602
  Mapped trades: 9904/10580 (93.6%)
  ✓ Status: EXCELLENT

  Controller breakdown:
    - brigado-binance-btcbrl-3: 2942 trades
    - brigado-binance-btcbrl-5: 2609 trades
    - brigado-binance-btcbrl-4: 1763 trades
    - brigado-binance-btcbrl-1: 1462 trades
    - brigado-binance-btcbrl-2: 1128 trades

Testing: pmm-mister-all-20260303-011107.sqlite
  Trades: 1380
  Executors (with PnL): 498
  Executors with order_ids: 498
  Total order mappings: 996
  Mapped trades: 1184/1380 (85.8%)
  ✓ Status: GOOD

  Controller breakdown:
    - brigado-binance-21-2: 283 trades
    - brigado-binance-21-3: 259 trades
    - brigado-binance-21-1: 253 trades
    - brigado-binance-18-1: 245 trades
    - brigado-binance-21-4: 144 trades

MAPPING VERIFICATION SUMMARY
Databases tested: 2
Total trades: 11960
Total mappe

## Usage Notes

### Running the Fetch
Simply run all cells to fetch the latest data from the server. Files are stored directly in:
- `data/live_databases/<bot_name>/data/*.sqlite`
- `data/live_databases/<bot_name>/conf/controllers/*.yml`

**Old files are automatically replaced** when you run the fetch again.

### Accessing Fetch Logs
```python
from pathlib import Path
import json

# List all fetch logs
base_path = Path("/Users/tomasgaudino/PycharmProjects/quants-lab/research_notebooks/brigado_v2/data/live_databases")
log_files = sorted(base_path.glob("fetch_log_*.json"), reverse=True)

# Load most recent fetch log
latest_log = log_files[0]
with open(latest_log, 'r') as f:
    fetch_data = json.load(f)

print(f"Last fetch: {fetch_data['timestamp']}")
print(f"Total files replaced: {fetch_data['total_replaced_files']}")
```

### Accessing Bot Databases
```python
# List all bot instances
bot_instances = [d for d in base_path.iterdir() if d.is_dir()]

# Access specific bot database
bot_name = "pmm-mister-all-20260303-011107"
db_path = base_path / bot_name / "data" / f"{bot_name}.sqlite"
```

### Troubleshooting

**SSH Connection Issues:**
- Verify SSH host alias: `ssh brigado`
- Check SSH config in `~/.ssh/config`
- Ensure SSH key authentication is set up

**Permission Errors:**
- Verify you have read access to remote directories
- Check file permissions on remote server

**Timeout Errors:**
- Large databases may take time to download
- Adjust timeout values in the code if needed
- Check network connectivity

### Next Steps

After fetching the databases, you can analyze them using the main `performance.ipynb` notebook by updating the `DB_NAME` parameter to point to one of the fetched databases.